# CT Evaluation — All Methods
Interactive evaluation notebook.  Run **after** training all three models.

This notebook calls the same functions as `ct/evaluate_all.py` but lets you
inspect intermediate outputs and customise plots interactively.

**Prerequisites:** checkpoint files in `ct/weights/`

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))
from config import CT, FISTA_NET, ISTA_NET, FBPCONVNET, CLASSICAL, EVAL, DEVICE, CT_WEIGHTS_DIR, CT_DATA_DIR

import numpy as np
import torch
import matplotlib.pyplot as plt

print("Device:", DEVICE)
print("Weights dir:", CT_WEIGHTS_DIR)
print("Checkpoints found:")
for f in sorted(CT_WEIGHTS_DIR.glob("*.pth")):
    print(f"  {f.name}")


## 1. Load Best Checkpoints

In [ ]:
import sys
sys.path.insert(0, os.path.abspath('..'))
from evaluate_all import find_best_checkpoint, load_fistanet, load_istanet, load_fbpconvnet
from dataset import build_ct_loaders, RadonOperator

_, _, test_loader = build_ct_loaders(data_root=CT_DATA_DIR, patch_size=None, batch_size=1)
fbp0, _, _  = next(iter(test_loader))
eff_size    = fbp0.shape[-1]
angles      = np.linspace(0.0, 180.0, CT["n_views"], endpoint=False)
radon_op    = RadonOperator(image_size=eff_size, n_views=CT["n_views"])

fista_model = load_fistanet(find_best_checkpoint(CT_WEIGHTS_DIR, "fistanet"), eff_size, DEVICE)
ista_model  = load_istanet( find_best_checkpoint(CT_WEIGHTS_DIR, "istanet"),  eff_size, DEVICE)
fbpc_model  = load_fbpconvnet(find_best_checkpoint(CT_WEIGHTS_DIR, "fbpconvnet"), DEVICE)

print("All models loaded.")
print(f"  FISTA-Net params : {fista_model.n_parameters():,}")
print(f"  ISTA-Net  params : {ista_model.n_parameters():,}")
print(f"  FBPConvNet params: {fbpc_model.n_parameters():,}")


## 2. Run Full Evaluation

In [ ]:
import argparse
from evaluate_all import evaluate_all

class Args:
    fista_tv_iters = CLASSICAL["fista_tv_ct_iters"]
    n_display      = EVAL["n_display"]
    n_views        = CT["n_views"]

results = evaluate_all(
    fista_model, ista_model, fbpc_model,
    radon_op, test_loader, angles, Args(), DEVICE
)
print("Evaluation complete.")


## 3. Print Comparison Table

In [ ]:
from shared.metrics import print_results_table
print_results_table(results,
    title=f"CT Reconstruction ({CT['n_views']}-view Sparse, Mayo Clinic)")


## 4. Visual Reconstruction Comparison

In [ ]:
from evaluate_all import save_comparison_figure
from baselines     import fista_tv_ct
from train_fistanet import run_fista_ct
from train_istanet  import run_ista_ct
from shared.metrics import compute_metrics

display_rows = []
with torch.no_grad():
    for fbp, sino, gt in test_loader:
        if len(display_rows) >= EVAL["n_display"]:
            break
        fbp_np  = fbp.squeeze(1).numpy()
        sino_np = sino.squeeze(1).numpy()
        gt_np   = gt.squeeze(1).numpy()

        x_fista, _ = run_fista_ct(fista_model, fbp, sino, radon_op, DEVICE)
        x_ista,  _ = run_ista_ct( ista_model,  fbp, sino, radon_op, DEVICE)
        fbpc_pred  = fbpc_model(fbp.to(DEVICE)).squeeze(1).cpu().numpy()

        for i in range(fbp.shape[0]):
            if len(display_rows) >= EVAL["n_display"]: break
            fbp_i = fbp_np[i]; sino_i = sino_np[i]; gt_i = gt_np[i]
            tv_i  = fista_tv_ct(fbp_i, sino_i, angles)
            row = {"FBP": fbp_i, "FISTA-TV": tv_i,
                   "ISTA-Net":   x_ista[i, 0].cpu().numpy(),
                   "FBPConvNet": fbpc_pred[i],
                   "FISTA-Net":  x_fista[i, 0].cpu().numpy(),
                   "GT":         gt_i}
            row["metrics"] = {m: compute_metrics(row[m], gt_i)
                               for m in ["FBP","FISTA-TV","ISTA-Net","FBPConvNet","FISTA-Net"]}
            display_rows.append(row)

save_comparison_figure(display_rows, "ct_comparison_interactive.png")


## 5. Learned Parameter Schedules

In [ ]:
from evaluate_all import save_learned_params_figure
save_learned_params_figure(fista_model, "ct_learned_params_interactive.png")

mus, thetas, rhos = fista_model.get_learned_params()
print("Learned parameters per stage:")
print(f"  mu     : {[round(v,4) for v in mus]}")
print(f"  theta  : {[round(v,4) for v in thetas]}")
print(f"  rho    : {[round(v,4) for v in rhos]}")


## 6. Training Curves (from checkpoint history)

In [ ]:
import torch
fista_ckpt = find_best_checkpoint(CT_WEIGHTS_DIR, "fistanet")
ckpt_data  = torch.load(fista_ckpt, map_location="cpu", weights_only=False)
history    = ckpt_data.get("history", {})

if "train_loss" in history and "val_psnr" in history:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(history["train_loss"], "b-o", markersize=4)
    ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss")
    ax1.set_title("FISTA-Net Training Loss"); ax1.grid(alpha=0.3)
    ax2.plot(history["val_psnr"], "g-o", markersize=4)
    ax2.set_xlabel("Epoch"); ax2.set_ylabel("PSNR (dB)")
    ax2.set_title("FISTA-Net Val PSNR"); ax2.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig("ct_training_curves.png", dpi=120, bbox_inches="tight")
    plt.show()
else:
    print("No history found in checkpoint (re-save training with history dict).")


## 7. Save All Results

In [ ]:
from shared.metrics import save_results_csv, save_results_summary_csv
from config import CT_RESULTS_DIR

(CT_RESULTS_DIR / "tables").mkdir(parents=True, exist_ok=True)
(CT_RESULTS_DIR / "figures").mkdir(parents=True, exist_ok=True)

save_results_csv(results,         CT_RESULTS_DIR / "tables" / "ct_per_sample.csv")
save_results_summary_csv(results, CT_RESULTS_DIR / "tables" / "ct_summary.csv")

from evaluate_all import save_bar_chart, save_error_map_figure
save_bar_chart(results, CT_RESULTS_DIR / "figures" / "ct_metrics_barchart.png")
if display_rows:
    save_error_map_figure(display_rows, CT_RESULTS_DIR / "figures" / "ct_error_maps.png")

print("All results saved to", CT_RESULTS_DIR)
